# Rank-mean ensembling: +0.015 AUC for free, and why probability-mean is the wrong tool

Three models of mine plateaued within 0.001 of each other on the public leaderboard. Rank-averaging them
moved the score more than a week of architecture work had:

| submission | public LB |
|---|---|
| v2 — single sagittal series, ResNet-34, mean-pooled slices | 0.873 |
| v3 — three planes, per-label plane-mixture head | 0.874 |
| v4 — ConvNeXt-T, gated slice attention, EMA weights | 0.883 |
| **rank mean of v2 + v3 + v4** | **0.898** |

**+0.015 over the best member**, from arithmetic on predictions I already had — no training, one submission.
This notebook explains why the gain exists, why *rank* mean rather than probability mean, when it fails, and
gives drop-in code. Everything is CPU-only and reproducible here.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

PATH = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0])
train = pd.read_csv(os.path.join(PATH, 'train.csv'))
LABELS = list(train.columns[2:14])
ann = train[train[LABELS].notna().all(axis=1)].reset_index(drop=True)
truth = ann[LABELS].values.astype(int)
print(f'{len(ann)} annotated studies used as ground truth')

## 1. The metric only reads order

This competition scores macro ROC AUC. AUC is invariant under **any strictly increasing transform** of a
label's scores — multiply by 10, take a square root, apply a sigmoid twice: identical AUC. Three consequences
follow, and each one removes a decision:

* Calibration is worth nothing. Do not spend effort on it.
* Thresholds are worth nothing. Do not tune them.
* **Combining models should combine orderings, not magnitudes.**

That last point is the whole argument for rank mean. Demonstration on real data below: I take one label table
and rescale it three ways that leave every ordering intact.

In [ ]:
src = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)[0]
tbl = pd.read_csv(src).set_index('StudyInstanceUID')
P = tbl.loc[ann['StudyInstanceUID'], LABELS].values.astype(float)

variants = {'raw p': P,
            'p ** 3 (same order)': P ** 3,
            'p * 100 (same order)': P * 100,
            'logit-ish (same order)': np.log(np.clip(P, 1e-6, 1-1e-6) / (1 - np.clip(P, 1e-6, 1-1e-6)))}
for name, V in variants.items():
    m = np.mean([roc_auc_score(truth[:, j], V[:, j]) for j in range(12) if 0 < truth[:, j].sum() < len(truth)])
    print(f'{name:24s} macro AUC {m:.6f}')

Identical to six decimals. Now watch what happens when you *average* two of those representations — the
scale, which the metric ignores, suddenly decides the outcome.

In [ ]:
A = P                      # model A: well-spread probabilities
B = P ** 6                 # model B: same ordering, but crushed toward 0 (an under-confident model)

def macro(X):
    return np.mean([roc_auc_score(truth[:, j], X[:, j]) for j in range(12) if 0 < truth[:, j].sum() < len(truth)])

def rank_norm(X):
    return np.stack([rankdata(X[:, j]) / len(X) for j in range(X.shape[1])], axis=1)

# perturb each model differently so they are not literally identical rankings
rng = np.random.RandomState(0)
A_n = np.clip(A + rng.normal(0, 0.10, A.shape), 0, 1)
B_n = np.clip(B + rng.normal(0, 0.02, B.shape), 0, 1)

print(f'model A alone           {macro(A_n):.4f}')
print(f'model B alone           {macro(B_n):.4f}')
print(f'probability mean        {macro((A_n + B_n) / 2):.4f}')
print(f'RANK mean               {macro((rank_norm(A_n) + rank_norm(B_n)) / 2):.4f}')

Probability mean lets whichever model happens to output a wider numeric range dominate the sum — a property
the metric never asked about. Rank mean puts every member on the same [0, 1] scale by construction, so each
one contributes its *opinion about ordering* and nothing else.

**Rule:** for any rank-based metric (ROC AUC, MAP, NDCG), average ranks. For a metric that reads magnitudes
(log loss, RMSE, Brier), average the values — and calibrate.

## 2. Why blending helps at all: members must fail differently

An ensemble only gains where members disagree. My three models each *won* different findings — and the blend
beat all three on the labels where they split:

| finding | v2 | v3 | v4 | rank mean |
|---|---|---|---|---|
| Effusion | 0.771 | 0.829 | **0.893** | 0.850 |
| MCL | 0.789 | **0.846** | 0.844 | **0.865** |
| Fracture | 0.812 | 0.843 | 0.835 | **0.869** |
| Contusion | 0.881 | 0.893 | 0.892 | **0.909** |
| Medial OA | 0.912 | 0.941 | 0.935 | **0.946** |
| ACL | **0.963** | 0.926 | 0.891 | 0.945 |

Note ACL: the blend is *worse* than v2 alone there, because v2 was genuinely better and the other two dragged
it down. That is the trade — you lose a little where one member is clearly best, and gain more where none is.
Across twelve labels it netted +0.015.

Correlation is the diagnostic. Members correlated above ~0.95 add almost nothing; the useful range is roughly
0.7-0.9 — different enough to disagree, good enough to be worth listening to.

In [ ]:
# how much diversity is enough? simulate members with a controlled correlation to the truth-signal
def make_member(base, noise, seed):
    r = np.random.RandomState(seed)
    return np.clip(base + r.normal(0, noise, base.shape), 0, 1)

base = P
rows = []
for noise in [0.05, 0.15, 0.30]:
    for n_members in [1, 2, 3, 5, 8]:
        ms = [make_member(base, noise, s) for s in range(n_members)]
        blended = np.mean([rank_norm(m) for m in ms], axis=0)
        corr = np.mean([np.corrcoef(ms[0][:, j], ms[-1][:, j])[0, 1] for j in range(12)]) if n_members > 1 else 1.0
        rows.append({'member noise': noise, 'members': n_members,
                     'mean pairwise corr': round(corr, 3), 'macro AUC': round(macro(blended), 4)})
div = pd.DataFrame(rows)
print(div.pivot(index='members', columns='member noise', values='macro AUC').to_string())

fig, ax = plt.subplots(figsize=(7, 4))
for noise, g in div.groupby('member noise'):
    ax.plot(g['members'], g['macro AUC'], 'o-', label=f'member noise {noise}')
ax.set_xlabel('number of members'); ax.set_ylabel('macro AUC of rank mean')
ax.set_title('Diminishing returns: most of the gain arrives by 3-5 members')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

Most of the benefit lands by three to five members, and noisier (more diverse) members benefit more from
being blended — which is why adding a *weaker but different* model often helps while adding a near-copy of
your best one does not.

## 3. Drop-in code

Rank-normalise per label, average, write. That is the entire technique.

In [ ]:
def rank_blend(prediction_frames, weights=None):
    """prediction_frames: list of DataFrames indexed by StudyInstanceUID, one column per label.

    Ranks are computed per label per member, so members never need to share a calibration.
    """
    ids = prediction_frames[0].index
    cols = list(prediction_frames[0].columns)
    if weights is None:
        weights = [1.0] * len(prediction_frames)
    weights = np.array(weights, dtype=float); weights /= weights.sum()

    out = np.zeros((len(ids), len(cols)))
    for w, df in zip(weights, prediction_frames):
        arr = df.loc[ids, cols].values.astype(float)
        out += w * np.stack([rankdata(arr[:, j]) / len(arr) for j in range(arr.shape[1])], axis=1)
    return pd.DataFrame(out, index=ids, columns=cols)

# toy demonstration with three perturbed members
members = [pd.DataFrame(make_member(P, 0.15, s), index=ann['StudyInstanceUID'], columns=LABELS)
           for s in (1, 2, 3)]
blend = rank_blend(members)
print('members :', [round(macro(m.values), 4) for m in members])
print('blend   :', round(macro(blend.values), 4))

## Practical notes and honest caveats

* **Equal weights are a strong default.** I measured a tuned weighting at 0.8779 versus 0.8769 for equal —
  a 0.001 difference on 58 studies, which is noise. Tuned weights on a small validation set are a good way to
  overfit; I submitted equal.
* **A weaker member can still help.** My fourth model scores lower alone but reads Synovitis at 0.78 where
  the others manage 0.62-0.72. Blending is about coverage, not about a leaderboard of members.
* **Decode once, predict many.** My three members need three different preprocessings; reading each study's
  DICOMs once and transforming three ways kept inference inside the runtime limit.
* **This is not free of risk.** Blending models tuned against the public leaderboard can inherit that
  overfitting. Blend models selected on your own holdout instead.

If this is useful, an upvote helps others find it — and I would genuinely like to hear counter-results where
probability mean beat rank mean on an AUC metric.